# Block 1: Synthetic OMOP-Style Healthcare Batch Pipeline

End-to-end walkthrough of the pipeline in five stages:

| Stage | What happens |
|---|---|
| 1 | Read raw CSVs (6 OMOP-style tables, ~1.5% dirty rows injected) |
| 2 | Validate raw — detection only, violations logged |
| 3 | Clean — drop dirty rows, compare before/after counts |
| 4 | Validate cleaned — hard gate, must pass with 0 violations |
| 5 | Build `analytic_person` and write partitioned Parquet |

In [ ]:
import sys
from pathlib import Path

# Make sure src/ is importable whether Jupyter is launched from the project
# root or from inside notebooks/
PROJECT_ROOT = Path(".").resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import io_utils, transforms, validations
from src.config import PROCESSED_DIR, RAW_DIR

spark = io_utils.get_spark_session("block1-demo")
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "ready")

---
## 0. Raw data overview

Read all six tables from `data/raw/`.  
If the CSVs are missing, run `python -m src.generator` from the project root first.

In [ ]:
person      = io_utils.read_person(spark)
visit       = io_utils.read_visit_occurrence(spark)
condition   = io_utils.read_condition_occurrence(spark)
drug        = io_utils.read_drug_exposure(spark)
measurement = io_utils.read_measurement(spark)
note        = io_utils.read_note(spark)

table_counts = {
    "person":               person.count(),
    "visit_occurrence":     visit.count(),
    "condition_occurrence": condition.count(),
    "drug_exposure":        drug.count(),
    "measurement":          measurement.count(),
    "note":                 note.count(),
}

pd.DataFrame(
    [{"table": t, "raw_rows": c} for t, c in table_counts.items()]
).set_index("table")

In [ ]:
print("person schema")
person.printSchema()

In [ ]:
person.limit(5).toPandas()

In [ ]:
visit.limit(5).toPandas()

---
## Stage 1 — Validate Raw (detection only)

Every dirty row injected by the generator should appear here.  
The pipeline logs violations but does **not** stop at this stage.

In [ ]:
raw_results = validations.validate_all(person, visit, condition, drug, measurement, note)

raw_df = pd.DataFrame([
    {"table": r.table, "check": r.check, "violations": r.count}
    for r in raw_results
])

total_violations = raw_df["violations"].sum()
print(f"Total violations in raw data: {total_violations}")
raw_df[raw_df["violations"] > 0].reset_index(drop=True)

In [ ]:
# All checks including the passing ones
raw_df.set_index(["table", "check"])

---
## Stage 2 — Clean

`clean_all()` drops dirty rows in dependency order:
1. Clean `person` (nulls + dupes)
2. Clean `visit_occurrence` against cleaned person (orphan FK drop)
3. Clean the remaining four tables

Returns `CleanedTables` and `CleaningMetrics` (before/after counts).

In [ ]:
tables, metrics = transforms.clean_all(person, visit, condition, drug, measurement, note)

metrics_df = pd.DataFrame([
    {
        "table":   t,
        "before":  metrics.before[t],
        "after":   metrics.after[t],
        "dropped": metrics.before[t] - metrics.after[t],
        "drop_%":  round(100 * (metrics.before[t] - metrics.after[t]) / metrics.before[t], 2),
    }
    for t in metrics.before
]).set_index("table")

metrics_df

---
## Stage 3 — Validate Cleaned (hard gate)

Re-run the same checks on the cleaned tables.  
If any violations remain the pipeline raises `PipelineValidationError`.  
We expect **0 violations** here.

In [ ]:
clean_results = validations.validate_all(
    tables.person, tables.visit, tables.condition,
    tables.drug, tables.measurement, tables.note,
)

clean_df = pd.DataFrame([
    {"table": r.table, "check": r.check, "violations": r.count}
    for r in clean_results
])

remaining = clean_df["violations"].sum()
if remaining == 0:
    print("Hard gate PASSED — 0 violations in cleaned tables")
else:
    print(f"Hard gate FAILED — {remaining} violations remain")
    print(clean_df[clean_df["violations"] > 0])

---
## Stage 4 — Build `analytic_person`

Joins and aggregates the six cleaned tables into one row per person, with:
- `age` (as of 2025-01-01)
- `year_of_birth_band` (decade, e.g. `"1980s"`) — used as the Parquet partition key
- visit counts by type (outpatient / inpatient / ER)
- condition count + diabetes / hypertension flags
- drug exposure count
- measurement count + latest HbA1c + latest systolic BP

In [ ]:
analytic = transforms.build_analytic_person(
    tables.person, tables.visit, tables.condition,
    tables.drug, tables.measurement,
)

print(f"analytic_person row count: {analytic.count()}")
analytic.printSchema()

In [ ]:
analytic.limit(10).toPandas()

---
## Stage 5 — Write Parquet

Writes `analytic_person` to `data/processed/`, partitioned by `year_of_birth_band`.

In [ ]:
io_utils.write_parquet(analytic, PROCESSED_DIR, partition_by=["year_of_birth_band"])

partitions = sorted(p.name for p in PROCESSED_DIR.iterdir() if p.is_dir())
print(f"Written to: {PROCESSED_DIR}")
print(f"Partitions ({len(partitions)}):")
for p in partitions:
    print(" ", p)

---
## Pipeline metrics

The pipeline writes `data/processed/pipeline_metrics.json` on every run with deterministic row counts, validation results, and stage timings.  
Since the data is generated with a fixed seed, the row counts should match exactly across runs.

In [ ]:
import json

metrics_path = PROCESSED_DIR / "pipeline_metrics.json"
expected_path = PROJECT_ROOT / "data" / "sample" / "expected_metrics.json"

if not metrics_path.exists():
    print(f"No metrics file found at {metrics_path}")
    print("Run 'python -m src.pipeline' to generate it.")
else:
    with open(metrics_path) as f:
        actual = json.load(f)

    # Display row counts
    print("Row counts (raw → cleaned → dropped):")
    counts = actual["row_counts"]
    count_rows = []
    for t in counts["raw"]:
        count_rows.append({
            "table": t,
            "raw": counts["raw"][t],
            "cleaned": counts["cleaned"][t],
            "dropped": counts["dropped"][t],
        })
    count_rows.append({
        "table": "analytic_person",
        "raw": "—",
        "cleaned": counts["analytic_person"],
        "dropped": "—",
    })
    display(pd.DataFrame(count_rows).set_index("table"))

    print(f"\nStage timings (seconds):")
    display(pd.DataFrame([actual["timings_seconds"]]).T.rename(columns={0: "seconds"}))

    # Compare against committed reference
    if expected_path.exists():
        with open(expected_path) as f:
            expected = json.load(f)

        mismatches = []
        for section in ("raw", "cleaned", "dropped"):
            for t, exp_val in expected["row_counts"][section].items():
                act_val = counts[section].get(t)
                if act_val != exp_val:
                    mismatches.append(f"  {section}.{t}: expected {exp_val}, got {act_val}")

        exp_ap = expected["row_counts"]["analytic_person"]
        if counts["analytic_person"] != exp_ap:
            mismatches.append(f"  analytic_person: expected {exp_ap}, got {counts['analytic_person']}")

        if mismatches:
            print("\n⚠ Row count mismatches vs expected_metrics.json:")
            print("\n".join(mismatches))
        else:
            print("\n✓ All row counts match data/sample/expected_metrics.json")
    else:
        print(f"\nNo reference file at {expected_path} — skipping comparison.")

---
## Explore — analytic_person

Quick looks at the final dataset.

In [ ]:
# Row count per decade band
(
    analytic
    .groupBy("year_of_birth_band")
    .count()
    .orderBy("year_of_birth_band")
    .toPandas()
    .set_index("year_of_birth_band")
)

In [ ]:
# Diabetes and hypertension prevalence
from pyspark.sql import functions as F

total = analytic.count()
prev = analytic.agg(
    F.sum(F.col("has_diabetes").cast("int")).alias("diabetes"),
    F.sum(F.col("has_hypertension").cast("int")).alias("hypertension"),
).collect()[0]

pd.DataFrame([
    {"condition": "diabetes",     "n": prev["diabetes"],     "prevalence_%": round(100 * prev["diabetes"]     / total, 1)},
    {"condition": "hypertension", "n": prev["hypertension"], "prevalence_%": round(100 * prev["hypertension"] / total, 1)},
])

In [ ]:
# Latest HbA1c summary stats (persons who have a measurement)
(
    analytic
    .filter(F.col("latest_hba1c").isNotNull())
    .select(
        F.count("*").alias("n"),
        F.round(F.mean("latest_hba1c"), 2).alias("mean"),
        F.round(F.stddev("latest_hba1c"), 2).alias("std"),
        F.round(F.min("latest_hba1c"), 2).alias("min"),
        F.round(F.max("latest_hba1c"), 2).alias("max"),
    )
    .toPandas()
)

In [ ]:
# Latest systolic BP summary stats
(
    analytic
    .filter(F.col("latest_systolic_bp").isNotNull())
    .select(
        F.count("*").alias("n"),
        F.round(F.mean("latest_systolic_bp"), 1).alias("mean"),
        F.round(F.stddev("latest_systolic_bp"), 1).alias("std"),
        F.round(F.min("latest_systolic_bp"), 1).alias("min"),
        F.round(F.max("latest_systolic_bp"), 1).alias("max"),
    )
    .toPandas()
)

In [ ]:
import matplotlib.pyplot as plt

# Patients per decade band
band_df = (
    analytic
    .groupBy("year_of_birth_band")
    .count()
    .orderBy("year_of_birth_band")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(band_df["year_of_birth_band"], band_df["count"])
ax.set_xlabel("Decade band")
ax.set_ylabel("Patients")
ax.set_title("Patients per decade band")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Diabetes vs hypertension prevalence
conditions = ["Diabetes", "Hypertension"]
counts = [prev["diabetes"], prev["hypertension"]]

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(conditions, counts)
ax.set_ylabel("Patients")
ax.set_title("Chronic condition prevalence")
for i, c in enumerate(counts):
    ax.text(i, c + 50, str(c), ha="center")
plt.tight_layout()
plt.show()

In [ ]:
spark.stop()
print("Done.")